In [ ]:
# 질병 단백질 ID 목록 추출
# labeled_proteins = set(df['uniprotAccession'].dropna().unique())

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split

# ✅ GCN 모델 (multi-label)
class GCN_MultiLabel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return torch.sigmoid(x)  # sigmoid로 각 질병별 확률 출력

# ✅ 라벨 생성 (multi-label): 단백질 → 질병 매핑 (0/1)
disease_ids = sorted(df['Disease ID'].dropna().unique().tolist())
disease_to_idx = {d: i for i, d in enumerate(disease_ids)}

y_multi = torch.zeros((len(proteins), len(disease_ids)))
for i, prot in enumerate(proteins):
    related_diseases = df[df['UniProt_ID'] == prot]['Disease ID'].dropna().unique()
    for d in related_diseases:
        if d in disease_to_idx:
            y_multi[i][disease_to_idx[d]] = 1

# 노드 인덱스 기준으로 train/test 나누기
idx = list(range(data.num_nodes))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask

# 하이퍼파라미터
in_channels = data.num_node_features
hidden_channels = 64
out_channels = len(torch.unique(data.y))

num_classes = len(torch.unique(data.y))
model = GCN_MultiLabel(
    in_channels=data.num_node_features,
    hidden_channels=64,
    out_channels=y_multi.shape[1]
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# 학습 루프
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()

    out = model(data)  # shape: [num_nodes, num_diseases]
    loss = loss_fn(out[data.train_mask], y_multi[data.train_mask])
    loss.backward()
    optimizer.step()

    # 평가
    model.eval()
    with torch.no_grad():
        pred = (out[data.test_mask] > 0.5).int()
        true = y_multi[data.test_mask].int()
        acc = (pred == true).float().mean()
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Acc: {acc:.4f}")

In [ ]:
def predict_node_multilabel(protein_id, model, data, protein_to_idx, idx_to_disease, threshold=0.5):
    model.eval()
    idx = protein_to_idx.get(protein_id)
    if idx is None:
        print("❌ Unknown protein ID:", protein_id)
        return [], []

    with torch.no_grad():
        output = model(data)[idx]
        probs = output.tolist()
        predicted = [idx_to_disease[i] for i, p in enumerate(probs) if p >= threshold]
        return predicted, probs

In [ ]:
protein_id = 'Q9HAZ2'
predicted_diseases, probs = predict_node_multilabel(protein_id, model, data, protein_to_idx, idx_to_disease)

print(f"🔍 Protein: {protein_id}")
print(f"🎯 예측된 질병 ID들: {predicted_diseases}")
print(f"📊 확률 분포 (상위 5개): {sorted(probs, reverse=True)[:5]}")

In [ ]:
predicted_disease_id = idx_to_disease[27]
print("예측 질병 ID:", predicted_disease_id)